# Lección 2.5 — División de datos: entrenamiento, validación y prueba

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 1.5 (ciclo de vida de un proyecto) y 2.3 (pipelines, de pasada)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar para qué sirve cada conjunto: **entrenamiento**, **validación** y **prueba**.
- Demostrar por qué evaluar con los mismos datos con los que se entrenó engaña.
- Entender por qué elegir modelos mirando el conjunto de prueba infla los resultados.
- Usar `train_test_split` con `test_size`, `stratify` y `random_state`, y saber cuándo **no** revolver los datos.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_classification, load_breast_cancer

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: estudiar con el examen

Una profesora quiere saber si sus alumnos **entendieron** matemáticas. Imagina tres formas de hacerlo:

1. Les da 50 ejercicios para practicar y luego los evalúa **con esos mismos 50 ejercicios**.
2. Les da los 50 ejercicios y evalúa con **ejercicios nuevos** que nunca vieron.
3. Igual que la 2, pero además les permite hacer **simulacros** de examen para decidir cómo estudiar, y el examen real lo guarda **bajo llave** hasta el final.

Con la opción 1, un alumno que **memorizó** las respuestas saca 10 sin haber entendido nada. Solo los ejercicios **nuevos** miden si aprendió.

En ML pasa exactamente lo mismo:

| En la escuela | En ML | Para qué sirve |
|---|---|---|
| Ejercicios de práctica | **Conjunto de entrenamiento** (*train*) | El modelo aprende de ellos |
| Simulacros | **Conjunto de validación** (*validation*) | **Tú** decides cosas: qué modelo, qué hiperparámetros |
| Examen final bajo llave | **Conjunto de prueba** (*test*) | Medir, **una sola vez**, qué tan bien funcionará con datos nuevos |

### ✋ Ejercicio 1 — Calentamiento
Un alumno tuvo 10 en los ejercicios de práctica, 8 en los simulacros y 7.5 en el examen final. Otro tuvo 10 en práctica, 5 en simulacros y 5 en el final. ¿Cuál "memorizó"? ¿Cuál de las tres notas dice mejor lo que sabe cada uno?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El segundo memorizó: 10 en lo que ya había visto y 5 en lo nuevo. La nota que mejor dice lo que saben es la del **examen final** (datos nunca vistos). La de práctica es la menos fiable: casi cualquiera saca buena nota en lo que ya vio.

Esa diferencia grande entre "práctica" y "examen" es la firma del **sobreajuste** (Lección 4.4).
</details>

---
## 2. 📖 El concepto: separar antes de aprender

La herramienta básica es `train_test_split`: revuelve las filas al azar y las reparte en dos grupos.

### 🔮 Ejercicio 2 — ¿Cuántas filas?
Tenemos 1,000 clientes y pedimos `test_size=0.2`. Antes de ejecutar: ¿cuántas filas tendrá cada grupo?

✍️ *Mi apuesta:* entrenamiento ___ · prueba ___

In [ ]:
X, y = make_classification(n_samples=1000, n_features=10, n_informative=4, flip_y=0.15, random_state=0)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)   # ✏️ prueba test_size=0.3
print("Entrenamiento:", X_tr.shape, "· Prueba:", X_te.shape)

<details><summary><b>👀 Ver respuesta</b></summary>

**800 y 200.** `test_size=0.2` significa "el 20 % para prueba". Con `0.3` serían 700 y 300.

`random_state=42` fija el "barajeo": con el mismo número, siempre salen las mismas filas en cada grupo. Así cualquiera puede repetir tu experimento. (El 42 no tiene nada de especial; es una broma de programadores.)
</details>

### 🔮 Ejercicio 3 — El memorizador
Un **KNN con 1 vecino** (Lección 5.3) predice copiando la etiqueta del ejemplo de entrenamiento más parecido. Es el alumno que memoriza. Estos datos tienen un **15 % de etiquetas con ruido** (Lección 2.2).

Antes de ejecutar: ¿qué exactitud tendrá en **entrenamiento**? ¿Y en **prueba**?

✍️ *Mi apuesta:*

In [ ]:
memorizador = KNeighborsClassifier(n_neighbors=1).fit(X_tr, y_tr)
print(f"Exactitud en ENTRENAMIENTO: {memorizador.score(X_tr, y_tr):.0%}")
print(f"Exactitud en PRUEBA       : {memorizador.score(X_te, y_te):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**100 % en entrenamiento**: cada ejemplo de entrenamiento es su propio vecino más cercano, así que el modelo "recuerda" su etiqueta (incluido el ruido). En **prueba** cae a ≈74 %.

Si solo hubieras mirado el entrenamiento, habrías anunciado un modelo perfecto. **Evaluar con los datos de entrenamiento no mide nada útil.**
</details>

---
## 3. 📖 ¿Por qué no basta con dos conjuntos? La trampa de mirar la prueba

Supón que pruebas 100 modelos distintos, evalúas cada uno con el conjunto de prueba y te quedas con el mejor. Suena razonable… pero acabas de **usar el examen para estudiar**. El ganador será, en parte, el que tuvo **más suerte** con esas preguntas concretas.

Vamos a demostrarlo de la forma más clara posible: con datos donde **no hay nada que aprender**. Las etiquetas son **aleatorias**, así que ningún modelo puede superar el 50 % de verdad.

### 🔮 Ejercicio 4 — 100 intentos con datos sin sentido
Entrenaremos 100 árboles distintos, mediremos cada uno en un conjunto de prueba de 60 ejemplos y nos quedaremos con el mejor. Luego mediremos a ese "campeón" con 10,000 ejemplos **nuevos**.

Antes de ejecutar: ¿qué exactitud tendrá el campeón en la prueba? ¿Y con los datos nuevos?

✍️ *Mi apuesta:* en prueba ___ · en datos nuevos ___

In [ ]:
rng = np.random.default_rng(0)
def datos_sin_sentido(n):
    return rng.normal(size=(n, 20)), rng.integers(0, 2, n)    # etiquetas al azar: no hay patrón

X_azar_tr, y_azar_tr = datos_sin_sentido(240)
X_azar_te, y_azar_te = datos_sin_sentido(60)
X_nuevos, y_nuevos = datos_sin_sentido(10_000)

mejor_puntaje, campeon = 0, None
for intento in range(100):                                   # ✏️ prueba con 10 o 1000 intentos
    modelo = DecisionTreeClassifier(max_depth=int(rng.integers(1, 8)),
                                    max_features=int(rng.integers(1, 21)),
                                    random_state=intento).fit(X_azar_tr, y_azar_tr)
    puntaje = modelo.score(X_azar_te, y_azar_te)
    if puntaje > mejor_puntaje:
        mejor_puntaje, campeon = puntaje, modelo

print(f"El campeón en el conjunto de prueba : {mejor_puntaje:.0%}")
print(f"El campeón con 10,000 datos nuevos  : {campeon.score(X_nuevos, y_nuevos):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El campeón saca ≈**65 %** en la prueba: parece haber aprendido algo. Con 10,000 datos nuevos, cae a ≈**50 %**, que es la verdad: **lanzar una moneda**.

Ese 65 % era **pura suerte** de haber elegido, entre 100 intentos, el que mejor "le atinó" a esas 60 preguntas. Cuantos más intentos, más se infla. ✏️ Pruébalo con 1,000.

**Moraleja:** si usas el conjunto de prueba para **elegir**, deja de servir para **medir**. Por eso necesitamos un tercer conjunto: el de **validación**, para elegir, y la prueba, guardada bajo llave, para medir **una sola vez** al final.
</details>

### 📖 Tres conjuntos
Un reparto típico es **60 % / 20 % / 20 %** (o 70/15/15 con muchos datos). Se hace con dos cortes:

In [ ]:
# Primer corte: separamos la PRUEBA y la guardamos bajo llave 🔒
X_resto, X_prueba, y_resto, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

# Segundo corte: del resto, sacamos la VALIDACIÓN
X_entren, X_valid, y_entren, y_valid = train_test_split(X_resto, y_resto, test_size=0.25, random_state=42)

print(f"Entrenamiento: {len(X_entren)} · Validación: {len(X_valid)} · Prueba: {len(X_prueba)}")

### 🔮 Ejercicio 5 — ¿Por qué 0.25?
En el segundo corte pusimos `test_size=0.25`, no 0.2. ¿Por qué? ¿Qué porcentaje del total termina en validación?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque el segundo corte se aplica sobre el **resto** (800 filas, el 80 % del total). El 25 % de 800 son **200**, que es el 20 % del total. Resultado: 600 / 200 / 200 = 60 % / 20 % / 20 %.
</details>

### 📖 El flujo completo
```
Todos los datos
   ├── 🔒 Prueba (20 %) ─────────────────────────────────────┐
   └── Resto (80 %)                                          │
         ├── Entrenamiento (60 %) → el modelo aprende        │
         └── Validación (20 %)    → TÚ eliges modelo e hp    │
                                                             ▼
   Al final: reentrenas al elegido con entrenamiento + validación
             y lo evalúas UNA VEZ con la prueba → ese es tu número honesto
```

> 💡 Con pocos datos, un solo conjunto de validación es poco fiable. La alternativa es la **validación cruzada** (Lección 4.6): rotar varias veces qué parte se usa para validar. Ya la viste de pasada (`cross_val_score`) en las lecciones anteriores.

---
## 4. 📖 Qué pasa si… el azar del reparto

Revolver al azar tiene una consecuencia: **otro reparto da otro resultado**. Con pocos datos, la diferencia puede ser grande.

### 🔮 Ejercicio 6 — La lotería del reparto
Vamos a repetir **20 veces** el mismo experimento (mismo modelo, mismos datos), cambiando solo el `random_state` de la división. Usaremos el conjunto de cáncer de mama (569 pacientes). Antes de ejecutar: ¿cuántos puntos de diferencia esperas entre el mejor y el peor resultado?

✍️ *Mi apuesta:*

In [ ]:
X_cancer, y_cancer = load_breast_cancer(return_X_y=True)

resultados = []
for semilla in range(20):
    Xa, Xb, ya, yb = train_test_split(X_cancer, y_cancer, test_size=0.2, random_state=semilla)
    resultados.append(DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xa, ya).score(Xb, yb))

print(f"Peor: {min(resultados):.1%} · Mejor: {max(resultados):.1%} · Promedio: {np.mean(resultados):.1%}")
plt.hist(resultados, bins=10, edgecolor="k"); plt.xlabel("exactitud en prueba")
plt.title("El mismo modelo, 20 repartos distintos"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Del peor (≈85 %) al mejor (≈96.5 %) hay unos **11 puntos**. ¡Con el mismo modelo! Si alguien te dice "mi modelo tiene 96 %", podría ser simplemente un reparto afortunado.

Por eso:
- Reporta siempre el `random_state` que usaste.
- No te emociones con diferencias de 1–2 puntos entre modelos evaluados con un solo reparto pequeño.
- Para decidir, usa validación cruzada (Lección 4.6), que promedia varios repartos.
</details>

---
## 5. 📖 Estratificar: que cada grupo se parezca al total

Si una clase es **rara** (por ejemplo, 5 % de fraudes), un reparto al azar puede dejar al conjunto de prueba con **muy pocos** o **ningún** caso raro. **Estratificar** (`stratify=y`) obliga a que cada grupo tenga la **misma proporción** de clases que el total.

### 🔮 Ejercicio 7 — ¿Cuántos fraudes en la prueba?
Tenemos 200 transacciones, de las cuales 10 son fraudes (5 %). Con un 20 % para prueba (40 transacciones), esperarías 2 fraudes. Antes de ejecutar: sin estratificar, ¿siempre habrá 2?

✍️ *Mi apuesta:*

In [ ]:
y_fraude = np.array([1] * 10 + [0] * 190)
X_fraude = np.arange(200).reshape(-1, 1)

for estratificar in [False, True]:
    conteos = []
    for semilla in range(10):
        _, _, _, y_prueba_f = train_test_split(X_fraude, y_fraude, test_size=0.2, random_state=semilla,
                                               stratify=y_fraude if estratificar else None)
        conteos.append(int(y_prueba_f.sum()))
    print(f"stratify={'y   ' if estratificar else 'None'} → fraudes en prueba en 10 repartos: {conteos}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Sin estratificar, los 10 repartos dejan entre **1** y **4** fraudes en la prueba (y con otros repartos podría tocar 0: ¡no podrías medir si tu modelo detecta fraudes!). Con 1 fraude, acertarlo o fallarlo cambia tu tasa de detección de 100 % a 0 %. Con `stratify=y`, **siempre 2**.

**Regla:** en clasificación, estratifica casi siempre, y **siempre** si hay una clase rara (Lección 2.7).
</details>

---
## 6. 📖 Cuándo NO revolver: el tiempo y los grupos

Revolver al azar supone que cada fila es **independiente**. Hay dos casos muy comunes en los que no lo es:

**1. Series de tiempo.** Si revuelves, el modelo entrena con datos de **marzo** y lo evalúas con datos de **febrero**: está "viendo el futuro". En la vida real siempre predices hacia adelante. La división correcta es **por fecha**:
```
|──────── entrenamiento (ene–sep) ────────|── validación (oct–nov) ──|── prueba (dic) ──|
```
Con `train_test_split` se consigue con `shuffle=False` (los datos deben estar ordenados por fecha), o simplemente cortando por una fecha.

**2. Grupos.** Si un mismo paciente tiene 10 visitas y algunas caen en entrenamiento y otras en prueba, el modelo puede **reconocer al paciente** en lugar de aprender de la enfermedad. Todas las filas de un mismo paciente (cliente, tienda, casa…) deben ir **al mismo lado**.

Ambos errores son formas de **fuga de datos**: la lección siguiente (2.6) está dedicada a ellas.

### ✋ Ejercicio 8 — ¿Cómo divides?
Para cada caso, elige: **al azar**, **al azar estratificado**, **por fecha** o **por grupo**.
1. Predecir el precio de una acción mañana.
2. Clasificar correos como spam (5 % son spam).
3. Diagnosticar una enfermedad con varias radiografías por paciente.
4. Predecir las ventas del próximo mes de cada tienda.
5. Predecir el precio de casas (una fila por casa, sin fechas).

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Por fecha** (serie de tiempo).
2. **Al azar estratificado** (clase rara).
3. **Por grupo** (paciente): todas las radiografías de un paciente juntas.
4. **Por fecha**: entrenas con meses pasados y pruebas con los más recientes. (Si además quieres saber cómo funciona con tiendas **nuevas**, también por grupo.)
5. **Al azar**: cada casa es independiente.
</details>

---
## 7. 🎛️ Las perillas de la división

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `test_size` | Fracción para prueba | Medición más fiable, pero el modelo aprende con menos datos | Más datos para aprender, pero medición más ruidosa | 0.2 (0.1 con muchísimos datos) |
| `stratify` | Mantener la proporción de clases | — | — | `stratify=y` en clasificación |
| `shuffle` | Revolver antes de dividir | — | `False` para series de tiempo (datos ordenados) | `True`, salvo con tiempo |
| `random_state` | La "semilla" del barajeo | — | — | Cualquier número fijo, para poder repetir el experimento |

## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Exactitud de entrenamiento ≈ 100 % y la de prueba mucho menor | El modelo memorizó (sobreajuste) | Simplificar el modelo (Lección 4.4); fíjate siempre en la de prueba |
| Los resultados cambian mucho al cambiar `random_state` | Pocos datos, conjunto de prueba pequeño | Validación cruzada (Lección 4.6) |
| El modelo empeora mucho al usarse de verdad | Elegiste mirando la prueba, o la división no imitaba la realidad | Validación para elegir; dividir por fecha o grupo si aplica |
| El conjunto de prueba no tiene casos de la clase rara | División sin estratificar | `stratify=y` |
| Resultados "demasiado buenos" con datos en el tiempo | Se revolvieron datos temporales | Dividir por fecha (Lección 2.6) |

### 🔮 Ejercicio 9 — Un buen hábito
Una científica de datos divide los datos, prueba 30 modelos en **validación**, elige el mejor y lo evalúa en **prueba**: 81 %. No le gusta, así que prueba 30 modelos más en validación, elige otro y lo vuelve a evaluar en prueba: 85 %. Lo reporta. ¿Es honesto ese 85 %?

<details><summary><b>👀 Ver respuesta</b></summary>

**No del todo.** Al volver a mirar la prueba y "reintentar" porque no le gustó el resultado, la prueba empezó a usarse para **decidir**. Es la trampa del Ejercicio 4 en cámara lenta: cada vistazo al examen lo "gasta" un poco. Lo honesto es reportar el primer resultado (81 %) o conseguir un **nuevo** conjunto de prueba.
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. **Entrenamiento** para aprender, **validación** para elegir, **prueba** para medir **una sola vez** al final.
2. Evaluar con los datos de entrenamiento engaña: un memorizador saca 100 %.
3. Elegir mirando la prueba **infla** el resultado, incluso con datos sin ningún patrón.
4. Divide con `train_test_split`: **estratifica** en clasificación y fija el `random_state`.
5. Con **tiempo** o **grupos**, no revuelvas al azar: divide por fecha o por grupo.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Para qué se usa el conjunto de **validación**?
- a) Para entrenar el modelo · b) Para elegir modelos e hiperparámetros · c) Para el resultado final que se reporta · d) Para nada; basta con prueba

**2.** Tienes 10,000 ejemplos y haces `train_test_split(..., test_size=0.3)`. ¿Cuántos quedan para entrenar?
- a) 3,000 · b) 7,000 · c) 10,000 · d) 6,000

**3.** Un modelo saca 99 % en entrenamiento y 70 % en prueba. ¿Qué cifra describe mejor cómo funcionará con clientes nuevos?
- a) 99 % · b) 70 % · c) El promedio, 84.5 % · d) Ninguna

**4.** ¿En cuál de estos casos es un **error** revolver los datos al dividir?
- a) Fotos de perros y gatos · b) Temperatura diaria de una ciudad · c) Encuesta de satisfacción · d) Precios de casas

**5.** Tus datos tienen un 2 % de casos positivos. ¿Qué opción de `train_test_split` es imprescindible?
- a) `shuffle=False` · b) `random_state=0` · c) `stratify=y` · d) `test_size=0.5`

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La validación es para **decidir**; la prueba, para **medir** al final.
2. **b)** El 70 % de 10,000.
3. **b)** La prueba son datos que el modelo no vio: es lo más parecido a clientes nuevos.
4. **b)** Es una serie de tiempo: hay que entrenar con el pasado y evaluar con el futuro.
5. **c)** Sin estratificar, el conjunto de prueba podría quedarse con poquísimos positivos.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Si el conjunto de prueba es tan valioso, ¿por qué no usar el 50 % de los datos para prueba? ¿Y por qué no el 1 %?

**B. Caso de análisis 💼** Una empresa de energía tiene 5 años de consumo eléctrico por hora y quiere predecir el consumo de mañana. Un analista divide al azar 80/20 y obtiene un error bajísimo. Explícale a su jefe, sin tecnicismos, por qué ese resultado probablemente es demasiado optimista y cómo dividirías tú.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Es un equilibrio: con 50 % para prueba la **medición** es muy fiable, pero el modelo aprende con la mitad de los datos y será peor. Con 1 %, el modelo aprende con casi todo, pero el examen es tan pequeño que su resultado depende mucho de la suerte (Ejercicio 6). Un 15–25 % suele ser un buen equilibrio; con millones de datos, un 1–5 % ya son muchísimos ejemplos.

**B.** "Al revolver, el modelo practicó con horas de **todos** los días, incluidas las del día siguiente y anterior a cada hora que luego le preguntamos. Es como predecir el clima de hoy a las 3 de la tarde sabiendo cuánto hizo a las 2 y a las 4: fácil. Pero mañana no tendremos el dato de las 4. Para medir de verdad, entrenaré con los primeros 4 años y medio y evaluaré con los últimos 6 meses, **en orden**, como se usará en la realidad."
</details>

### 10.3 🧩 Práctica integradora: "El flujo completo, sin trampas"

Volvemos al diagnóstico de cáncer de mama (569 pacientes). Usarás un **KNN** (con escalado, Lección 2.3) y elegirás su número de vecinos `k` siguiendo el flujo profesional: entrenamiento / validación / prueba.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ✏️ — Divide en tres conjuntos (60/20/20), estratificando.** Completa los dos `test_size` y ejecuta.

In [ ]:
X_c, y_c = load_breast_cancer(return_X_y=True)

X_resto, X_prueba, y_resto, y_prueba = train_test_split(
    X_c, y_c, test_size=0.2, stratify=y_c, random_state=0)            # ✏️ 🔒 prueba bajo llave
X_entren, X_valid, y_entren, y_valid = train_test_split(
    X_resto, y_resto, test_size=0.25, stratify=y_resto, random_state=0)  # ✏️

for nombre, yy in [("Entrenamiento", y_entren), ("Validación", y_valid), ("Prueba", y_prueba)]:
    print(f"{nombre:<14}: {len(yy):>3} pacientes · {yy.mean():.1%} benignos")

<details><summary><b>👀 Ver respuesta</b></summary>

341 / 114 / 114 pacientes, y los tres con ≈**62–63 %** de tumores benignos: la estratificación funcionó.
</details>

**Paso 2 🔮 — Elige `k` con validación.** Antes de ejecutar: con `k=1` (el memorizador), ¿qué exactitud esperas en entrenamiento? ¿Crees que será el mejor en validación?

In [ ]:
print(" k   entrenamiento   validación")
for k in [1, 3, 5, 9, 15, 25, 51, 101]:                  # ✏️
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_entren, y_entren)
    print(f"{k:>3}      {knn.score(X_entren, y_entren):.1%}          {knn.score(X_valid, y_valid):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `k=1` el entrenamiento da **100 %** (el memorizador)… ¡y en validación empata con los mejores (98.2 %)! ¿Sorprendido? Mira la escala: con 114 pacientes en validación, **un solo paciente** vale 0.9 puntos. Todos los `k` entre 1 y 15 están a uno o dos pacientes de distancia: es un **empate técnico**. Con `k` muy grande (101) el modelo se vuelve demasiado "promediador" y sí baja claramente (93 %).

Ante un empate, evita el extremo memorizador (`k=1`, el que más depende de cada paciente concreto) y elige uno del centro de la meseta: por ejemplo **k = 9**. Esta "lotería de un paciente" es justo la razón por la que en la práctica se usa validación cruzada (Lección 4.6).
</details>

**Paso 3 ✋ — Una tentación.** Un compañero propone: *"Para estar seguros, probemos también cada `k` contra la prueba y quedémonos con el que gane ahí"*. ¿Qué le respondes?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que entonces la prueba dejaría de ser un examen sorpresa: estaríamos **eligiendo** con ella, y su resultado saldría inflado (Ejercicio 4). La prueba se usa **una sola vez**, con el modelo ya elegido.
</details>

**Paso 4 ▶️ — Reentrena y abre la caja fuerte.** Con el `k` elegido, reentrenamos con **entrenamiento + validación** (más datos = mejor modelo) y evaluamos **una única vez** en la prueba.

In [ ]:
K_ELEGIDO = 9     # ✏️ pon aquí tu valor

final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=K_ELEGIDO)).fit(X_resto, y_resto)
print(f"Exactitud en PRUEBA (una sola vez): {final.score(X_prueba, y_prueba):.1%}")
print(f"Referencia 'decir siempre benigno': {y_prueba.mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**95.6 %** en prueba, contra 63.2 % de decir siempre "benigno". Es un poco menos que en validación (97.4 %), y es **normal**: el número de validación estaba ligeramente inflado porque lo usamos para elegir. El de prueba es el que reportas.
</details>

**Paso 5 ✋ — Repórtalo.** Escribe, en **dos frases** y para un médico que no sabe ML, qué tan bien funciona el modelo y cómo lo mediste.

✍️ *Mi reporte:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"El modelo clasificó correctamente cerca de 96 de cada 100 tumores en un grupo de 114 pacientes que **nunca** usamos para construirlo ni para ajustarlo. Lo comparamos con decir siempre 'benigno', que acertaría solo 63 de cada 100."*

(Para un médico, además, importará **qué tipo** de errores comete: no es lo mismo decir "benigno" a un tumor maligno que al revés. Eso lo verás en la Lección 4.2.)
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, guía de usuario, *Cross-validation: evaluating estimator performance* (incluye un diagrama del flujo entrenamiento/validación/prueba).
- 📺 **Video:** StatQuest, "Machine Learning Fundamentals: Cross Validation" (inglés, con subtítulos).
- 📘 **Lectura:** Google, *Machine Learning Crash Course*, sección "Training and Test Sets" (en español).

---
⬅️ **Anterior:** [2.4 Ingeniería de características: crear información útil](2.4-ingenieria-de-caracteristicas.ipynb) · ➡️ **Siguiente:** [2.6 Fuga de datos (*data leakage*): el error silencioso más caro](2.6-fuga-de-datos.ipynb)